# 18_SplitBonus: chia `tap` cho Dataset_bonus và gộp thành manifest_all.csv
- 215 video cũ giữ nguyên `tap` (không đụng `manifest.csv`).
- 90 video bonus chia 70/15/15 theo video, giữ tỉ lệ lớp, seed 42 (cùng cách cell 8 của 01). Kết quả mỗi lớp: 21 train / 4 val / 5 test.
- Cell 2 cho xem các video quay gần nhau (phiên quay). Nếu cần chia theo phiên thì đặt `GOP_PHIEN = True` ở cell 3.
- Ghi `manifest_bonus.csv` (đã có `tap`) và `manifest_all.csv` (215 + 90 = 305 dòng). Chạy lại ra đúng kết quả cũ.

In [1]:
#cell 1: đọc manifest cũ + manifest_bonus, kiểm tra cột, tap, file tồn tại
import re, numpy as np, pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")
old = pd.read_csv(BASE / "manifest.csv")
bonus = pd.read_csv(BASE / "manifest_bonus.csv", keep_default_na=False)

assert old.columns.tolist() == bonus.columns.tolist(), (old.columns.tolist(), bonus.columns.tolist())
assert old.tap.isin(["train", "val", "test"]).all(), "manifest.csv có dòng chưa có tap"
assert not set(old.path) & set(bonus.path), "path bonus trùng path cũ"
miss = [p for p in pd.concat([old.path, bonus.path]) if not (BASE / p).exists()]
assert not miss, f"thiếu {len(miss)} file, ví dụ: {miss[:3]}"

print("manifest cũ:", len(old), "|", old.tap.value_counts().to_dict())
print("manifest_bonus:", len(bonus), "|", bonus.lop.value_counts().to_dict())
print(old.path.iloc[0]); print(bonus.path.iloc[0])   # so kiểm tra cùng kiểu đường dẫn

manifest cũ: 215 | {'train': 151, 'test': 33, 'val': 31}
manifest_bonus: 90 | {'paper': 30, 'rock': 30, 'scissors': 30}
landmarks_120\Dataset_hand\paper\VID_20261002074120297.npz
landmarks_120\Dataset_bonus\paper\VID_20261024172018538.npz


In [2]:
#cell 2: xem các video bonus quay gần nhau (phiên quay) từ mốc giờ trong tên file, chưa chia gì
GAP_S = 60   # 2 video cách nhau quá 60 giây coi là sang phiên mới (đổi nếu cần)

def to_time(name):
    m = re.search(r"(\d{8})(\d{6})(\d{3})", name)
    return pd.to_datetime(m.group(1) + m.group(2) + m.group(3), format="%Y%m%d%H%M%S%f") if m else pd.NaT

bonus["gio"] = bonus.path.map(lambda p: to_time(Path(p).name))
print("Không đọc được giờ:", int(bonus.gio.isna().sum()), "video")
bonus = bonus.sort_values(["lop", "gio"]).reset_index(drop=True)
bonus["phien"] = bonus.groupby("lop").gio.transform(lambda s: (s.diff().dt.total_seconds() > GAP_S).cumsum())
ps = bonus.groupby(["lop", "phien"]).agg(so_video=("path", "count"), bat_dau=("gio", "min"), ket_thuc=("gio", "max"))
print(ps.to_string())

Không đọc được giờ: 0 video
                so_video                 bat_dau                ket_thuc
lop      phien                                                          
paper    0             1 2026-10-24 17:20:18.538 2026-10-24 17:20:18.538
         1             1 2026-10-24 17:26:35.730 2026-10-24 17:26:35.730
         2             1 2026-10-24 17:31:10.190 2026-10-24 17:31:10.190
         3             1 2026-10-24 17:34:18.364 2026-10-24 17:34:18.364
         4             1 2026-10-25 00:22:27.576 2026-10-25 00:22:27.576
         5             1 2026-10-25 00:27:27.377 2026-10-25 00:27:27.377
         6             1 2026-10-25 00:30:42.472 2026-10-25 00:30:42.472
         7             1 2026-10-25 00:33:21.620 2026-10-25 00:33:21.620
         8             1 2026-10-25 00:36:55.474 2026-10-25 00:36:55.474
         9             1 2026-10-25 00:41:12.747 2026-10-25 00:41:12.747
         10            1 2026-10-25 11:16:57.663 2026-10-25 11:16:57.663
         11            

In [3]:
#cell 3: chia tap 70/15/15 cho bonus (seed 42), giữ tỉ lệ lớp; ghi lại manifest_bonus.csv
GOP_PHIEN = False   # False: chia theo video như 01. True: mỗi phiên (cell 2) nằm trọn trong 1 tập

rng = np.random.default_rng(42)
bonus["tap"] = ""
for lop, g in bonus.groupby("lop"):
    n = len(g); n_tr, n_va = round(n * 0.70), round(n * 0.15)
    if not GOP_PHIEN:
        idx = rng.permutation(g.index.to_numpy())
        bonus.loc[idx[:n_tr], "tap"] = "train"
        bonus.loc[idx[n_tr:n_tr + n_va], "tap"] = "val"
        bonus.loc[idx[n_tr + n_va:], "tap"] = "test"
    else:
        target = {"train": n_tr, "val": n_va, "test": n - n_tr - n_va}; cur = dict.fromkeys(target, 0)
        for ph in rng.permutation(g.phien.unique()):
            ids = g.index[g.phien == ph]
            t = max(target, key=lambda k: target[k] - cur[k])   # tập đang thiếu nhiều nhất
            bonus.loc[ids, "tap"] = t; cur[t] += len(ids)

assert (bonus.tap != "").all()
bonus[old.columns].to_csv(BASE / "manifest_bonus.csv", index=False)
print("Cách chia:", "theo phiên" if GOP_PHIEN else "theo video")
print(bonus.pivot_table(index="lop", columns="tap", values="path", aggfunc="count")[["train", "val", "test"]])

Cách chia: theo video
tap       train  val  test
lop                       
paper        21    4     5
rock         21    4     5
scissors     21    4     5


In [4]:
#cell 4: gộp manifest_all.csv = manifest.csv (giữ nguyên) + manifest_bonus.csv, kiểm tra lại
allm = pd.concat([old, bonus[old.columns]], ignore_index=True)
assert len(allm) == len(old) + len(bonus) == allm.path.nunique()
assert allm.iloc[:len(old)].equals(old), "dòng cũ bị đổi"
allm.to_csv(BASE / "manifest_all.csv", index=False)

print("manifest_all.csv:", len(allm), "video | lưu tại:", BASE / "manifest_all.csv")
print(allm.pivot_table(index=["nguon", "lop"], columns="tap", values="path", aggfunc="count")[["train", "val", "test"]])
print(allm.tap.value_counts())

manifest_all.csv: 305 video | lưu tại: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\manifest_all.csv
tap                       train  val  test
nguon           lop                       
Dataset_bonus   paper        21    4     5
                rock         21    4     5
                scissors     21    4     5
Dataset_hand    paper        19    4     4
                rock         19    4     4
                scissors     19    4     4
dataset_labeled paper        34    7     7
                rock         30    6     7
                scissors     30    6     7
tap
train    214
test      48
val       43
Name: count, dtype: int64
